# 03 · Non-covalent binding modes

For a multi-fragment complex, rxembed discovers cooperative binding modes and searches each one. A mode
is a set of contacts, each with a distance window and any required direction.

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import Chem
import matplotlib.pyplot as plt

rx.set_verbose("INFO")

## `contacts='auto'`: discover, then search each grip

A bifunctional (iminophosphorane–thiourea) catalyst + a mandelamide substrate has many distinct grips. `contacts='auto'` returns an `EnsembleSet`, one candidate per discovered mode; a geometrically infeasible grip embeds to zero conformers and is skipped.

In [ ]:
bimp = "CC(CN=P(C)(C)C)NC(=S)Nc1ccccc1.NC(=O)C(O)c1ccccc1"
# stereo="free" avoids multiplying each grip by the four undefined backbone stereoisomers.
rx.set_verbose("WARNING")  # suppress one identical seed summary per candidate
modes = rx.embed(bimp, contacts="auto", n=8, stereo="free")  # EnsembleSet: one Ensemble per grip
rx.set_verbose("INFO")
print(f"{len(modes)} distinct binding modes discovered\n")
shown = 0
for m in modes:
    best = m.mc(preset="rapid").minimize().prune()
    clean = [cid for cid, report in best.check().items() if report.ok()]
    if not clean:
        continue
    shown += 1
    print(f"grip {shown}: {len(clean):3d} clean confs  |  {m.tag.get('nci')}")
    if shown >= 4:
        break

## Beyond the H-bond: halogen, chalcogen, salt bridge

`nci_candidates` enumerates directional contacts with an orientation angle, so the seeded geometry is real.
σ-holes appear only on a polarisable heavy donor (I, S), never an amine N or a C–F.

In [ ]:
systems = {
    "halogen bond  C-I···N": "FC(F)(F)C(F)(F)I.c1ccncc1",
    "chalcogen     S···O=C": "c1nc2ccccc2s1.O=C(C)C",
    "salt bridge   NH3+···-OOC": "CC(=O)[O-].CCC[NH3+]",
}
for name, smi in systems.items():
    cands = rx.nci_candidates(Chem.AddHs(Chem.MolFromSmiles(smi)))
    kinds = sorted({k.split(":")[0] for k in cands})
    print(f"  {name:24s}: kinds {kinds}, e.g. {list(cands)[0]}")

### π contacts: cation-π (requestable)

Ring contacts are omitted from `contacts='auto'` because a rough reference conformer produces many
candidates. Request them with `kinds=`. The cation–π example uses per-ring-atom distances to place the ion
over the ring centroid; it needs no separate angle.

In [ ]:
cands = rx.nci_candidates(Chem.AddHs(Chem.MolFromSmiles("C[N+](C)(C)C.c1ccccc1")), kinds=("CATPI", "CHPI"))
catpi = next(k for k in cands if k.startswith("CATPI"))
print("π contact:", catpi)
ens = rx.embed("C[N+](C)(C)C.c1ccccc1", contacts=cands[catpi], n=8).mc(preset="rapid").minimize().prune()
ring = [a.GetIdx() for a in ens.mol.GetAtoms() if a.GetIsAromatic()]
N = ens.mol.GetSubstructMatch(Chem.MolFromSmarts("[N+]"))[0]
import numpy as np

sep = np.mean(
    [
        float(
            np.linalg.norm(
                ens.mol.GetConformer(c).GetPositions()[N] - ens.mol.GetConformer(c).GetPositions()[ring].mean(0)
            )
        )
        for c in ens.ids
    ]
)
for report in ens.check().values():
    print(report.summary())
print(f"{len(ens)} conformers; N⁺···ring-centroid {sep:.2f} Å (cation-π)")
xyzrender.render(ens.mol, no_hy=True)

The halogen-bonded complex embeds inside its σ-hole distance window and passes the geometry gate.

In [ ]:
smi = "FC(F)(F)C(F)(F)I.c1ccncc1"
cands = rx.nci_candidates(Chem.AddHs(Chem.MolFromSmiles(smi)))
xb_contact = cands[next(k for k in cands if k.startswith("XB:"))]  # the I···N σ-hole, not halogen-π
xb = rx.embed(smi, contacts=xb_contact, n=8).mc(preset="ensemble").minimize().prune()
lo, hi = next(iter(xb_contact.distances.values()))
print(f"I···N realised: {xb.measure(('I', 'n'))['mean']:.2f} Å  ({len(xb)} confs, window {lo:.1f}-{hi:.1f})")
for report in xb.check().values():
    print(report.summary())
xyzrender.render(xb.mol, no_hy=True)

## A bifurcated grip: the Schreiner thiourea

`nci_modes` groups both thiourea N–H donors onto one carbonyl acceptor automatically: the bifurcated clamp. `binding_modes()` reports which contact patterns the search sampled.

In [ ]:
schr = "FC(F)(F)c1cc(cc(c1)C(F)(F)F)NC(=S)Nc1cc(cc(c1)C(F)(F)F)C(F)(F)F.CC(C)=O"
print("modes:", list(rx.nci_modes(Chem.AddHs(Chem.MolFromSmiles(schr)))))
mode = next(iter(rx.nci_modes(Chem.AddHs(Chem.MolFromSmiles(schr))).values()))
ens = rx.embed(schr, contacts=mode, n=10).mc(preset="ensemble").minimize().prune()
print(f"{len(ens)} conformers; inter-fragment patterns sampled:")
for sig, n in ens.binding_modes().most_common()[:4]:
    print(f"   {n:3d}: {list(sig) or ['(none)']}")

### The gate as an acceptance filter

The gate reports structural violations per conformer before scoring.

In [ ]:
ens.minimize()
total = ens.n
ens.filter("geometry")
print(f"accepted {ens.n}/{total} conformers")

## Bias the seed, let energy decide: `mc(explore=True)`

The seeded contact is a bias, not truth. `explore=True` pools searches with the grip held and released, while structural holds remain. GFN-FF then ranks the combined pool and requires `xtb` on `$XTB_EXE`.

In [ ]:
raw = rx.embed(schr, contacts=mode, n=8).mc(preset="rapid", explore=True).minimize()
pool = raw.filter("geometry").score("gfnff")
print(f"explore pool: {pool.n} conformers | GFN-FF ΔE 0-{max(pool.relative().values()):.1f} kcal/mol")
print(f"energy-chosen best grip: {pool.lowest(1).binding_modes().most_common(1)}")
pool.landscape("tsne", color="energy")  # coloured by the real energy now, not just cluster shape
plt.title("Schreiner·acetone; explore pool by GFN-FF energy")
plt.show()

In [ ]:
xyzrender.render(pool.lowest(2).mol, no_hy=True)